- In __Linear Regression__ we had:
$$
 y = mx + b
$$

- In __Logistic Regression__, instead of returning a value, we return a probability and for that we use __Sigmoid Function__:
$$
 y = \frac{1}{1 + \exp{(-x)}}
 \newline
 y =  \frac{1}{1 + \exp{(-mx + b)}}
$$

- Instead of using the __Mean Squared Error__, we use __Cross Entropy__:
$$
J(w, b) = J(\theta) = \frac{1}{N} \sum_{i=1}^{n}[y^i \log(h_{\theta}(x^i)) + (1 - y^i)\log(1 - h_{\theta}(x^i))]
$$

- In order to use the Gradient Descent algorithm, we need to calculate the gradient of this error function, in terms of the weight and the bias:
$$
\frac{dJ}{dw} = \frac{1}{N} \sum(2x_{i}(\hat y - y_{i}))
$$

$$
\frac{dJ}{db} = \frac{1}{N} \sum(2(\hat y - y_{i}))
$$

In [10]:
import numpy as np


def sigmoid(x):
    return 1/(1 + np.exp(-x))


class LogisticRegression():
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iters):
            linear_pred = np.dot(X, self.weights) + self.bias
            predictions = sigmoid(linear_pred)

            # calculate the gradients
            dw = 1/n_samples * np.dot(X.T, (predictions - y))
            db = 1/n_samples * np.sum(predictions - y)

            self.weights = self.weights - self.lr * dw
            self.bias = self.bias - self.lr * db

    def predict(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        y_pred = sigmoid(linear_pred)
        class_pred = [0 if y <= 0.5 else 1 for y in y_pred]

        return class_pred

In [11]:
from sklearn.model_selection import train_test_split
from sklearn import datasets
import matplotlib.pyplot as plt

bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Calculate the accuracy
def accuracy(y_pred, y_test):
    return np.sum(y_pred == y_test)/len(y_test)


acc = accuracy(y_pred=y_pred, y_test=y_test)
print(acc)

0.9473684210526315


/var/folders/v_/1czwrsjd5554z7_8tbs7r5jh0000gn/T/ipykernel_2320/1880958526.py:5: RuntimeWarning: overflow encountered in exp
  return 1/(1 + np.exp(-x))


---

Another Approach:

In [ ]:
import numpy as np


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def calculate_gradient(theta, X, y):
    m = y.size # no. of instances
    return (X.T @ (sigmoid(X @ theta) - y)) / m # @ is matrix multiplication symbol

def gradient_descent(X, y, alpha=0.1, num_iters=100, tol=1e-7):
    X_b = np.c_[np.ones((X.shape[0], 1)), X] # concatinated the bias dimension 
    theta = np.zeros(X_b.shape[1])

    for _ in range(num_iters):
        grad = calculate_gradient(theta, X_b, y)
        theta -= alpha * grad

        if np.linalg.norm(grad) < tol:
            break

    return theta

def predict_proba(X, theta):
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    return sigmoid(X_b @ theta) # the probability

def predict(X, theta, threshold=0.5):
    return (predict_proba(X, theta) >= threshold).astype(int)

In [18]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

theta_hat = gradient_descent(X_train_scaled, y_train, alpha=0.1)

y_pred_train = predict(X_train_scaled, theta_hat)
y_pred_test = predict(X_test_scaled, theta_hat)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(train_acc)
print(test_acc)

0.9824175824175824
0.9824561403508771
